# Market Position Based Pricing

Manual-run review script. For each (product, cohort), classify the SKU by its position in the market percentile band and yesterday's achievement bucket (`qty_ratio = yest_qty / p80_target`), then move N steps in the SKU's effective tier ladder per the agreed action matrix.

**No API push.** Outputs a timestamped Excel for review.

## Action matrix

| pos | low (<0.9) | mid (0.9-2.0) | high (>=2.0) |
|---|---|---|---|
| min | HOLD | HOLD | +2 |
| 25 | -1 | -1 | +1 |
| 50 | -2 | -1 | +1 |
| 75 | -3 | -3 | HOLD |
| max | -3 | -2 | HOLD |
| Target margin<target | -2 | -2 | +2 |
| Target margin>=target | -2 | -1 | HOLD |
| below_min | exact `commercial_min` | exact `commercial_min` | exact `commercial_min` |

Margin step bound: `0.1 * target_margin <= |delta_margin| <= 0.5 * target_margin`. Hard floor: `0.9 * wac_p` (commercial_min takes precedence).

In [ ]:
# =============================================================================
# CELL 1 - IMPORTS + SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake, get_snowflake_timezone
from constants import WAREHOUSE_MAPPING, COHORT_IDS

TIMEZONE = get_snowflake_timezone()
CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}  |  Snowflake TZ: {TIMEZONE}')

In [ ]:
# =============================================================================
# CELL 2 - SOURCE DATA LOAD
# Copies the loader pattern from manual_price_push.ipynb but with two important
# differences:
#   (a) current_price comes from cohort_pricing_changes only (DBDP is stopped).
#   (b) margin tiers are loaded via get_margin_tiers() so the Target case can
#       fall back to wac/(1 - margin_tier_X) prices.
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

# (a) current_price from cohort_pricing_changes (latest row per cohort, product, PU)
print('Loading current prices (cohort_pricing_changes)...')
CURRENT_PRICES_QUERY = f'''
WITH latest_changes AS (
    SELECT
        cpc.cohort_id,
        pup.product_id,
        pup.packing_unit_id,
        pup.basic_unit_count,
        cpc.price AS current_price
    FROM cohort_pricing_changes cpc
    JOIN packing_unit_products pup ON pup.id = cpc.product_packing_unit_id
    WHERE cpc.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
      AND pup.basic_unit_count = 1
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY cpc.cohort_id, pup.product_id, pup.packing_unit_id
        ORDER BY cpc.created_at DESC
    ) = 1
)
SELECT cohort_id, product_id, packing_unit_id, basic_unit_count, current_price
FROM latest_changes
WHERE basic_unit_count = 1
  AND ((product_id = 1309 AND packing_unit_id = 2) OR product_id <> 1309)
'''
df_prices = query_snowflake(CURRENT_PRICES_QUERY)
print(f'  Current prices: {len(df_prices)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins (brand x cat)...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading commercial min prices...')
COMMERCIAL_MIN_QUERY = f'''
WITH to_remove AS (
    SELECT check_date AS start_date,
           (check_date + INTERVAL '1 month') + 6 AS end_date
    FROM (
        SELECT CASE
                 WHEN DATE_PART('day', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) < 7
                 THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
                 ELSE DATE_FROM_PARTS(
                       YEAR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE),
                       MONTH(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE),
                       1)
               END AS check_date
    )
),
region_mapping AS (
    SELECT * FROM (VALUES
        ('Greater Cairo', 'Cairo'),
        ('Greater Cairo', 'Giza')
    ) x(region, main_region)
),
warehouse_mapping AS (
    SELECT * FROM (VALUES
        ('Cairo', 'Mostorod', 1, 700),
        ('Giza', 'Barageel', 236, 701),
        ('Giza', 'Sakkarah', 962, 701),
        ('Delta West', 'El-Mahala', 337, 703),
        ('Delta West', 'Tanta', 8, 703),
        ('Delta East', 'Mansoura FC', 339, 704),
        ('Delta East', 'Sharqya', 170, 704),
        ('Upper Egypt', 'Assiut FC', 501, 1124),
        ('Upper Egypt', 'Bani sweif', 401, 1126),
        ('Upper Egypt', 'Menya Samalot', 703, 1123),
        ('Upper Egypt', 'Sohag', 632, 1125),
        ('Alexandria', 'Khorshed Alex', 797, 702)
    ) x(region, warehouse_name, warehouse_id, cohort_id)
)
SELECT DISTINCT
    sku_id AS product_id,
    cohort_id,
    min_price AS commercial_min_price
FROM (
    SELECT mp.product_id AS sku_id, mp.region, min_price, wac1, created_at,
           MAX(created_at) OVER (PARTITION BY mp.product_id, mp.region) AS latest_date
    FROM finance.minimum_prices mp
    JOIN finance.all_cogs f ON f.product_id = mp.product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
    WHERE is_deleted = 'false'
      AND created_at BETWEEN (SELECT start_date FROM to_remove) AND (SELECT end_date FROM to_remove)
) comm
LEFT JOIN region_mapping rm ON comm.region = rm.region
LEFT JOIN warehouse_mapping wm ON wm.region = COALESCE(rm.main_region, comm.region)
WHERE created_at = latest_date
  AND min_price < wac1 * 1.5
'''
df_commercial_min = query_snowflake(COMMERCIAL_MIN_QUERY)
print(f'  Commercial min prices: {len(df_commercial_min)} rows')

print('Loading stocks (per warehouse, basic unit)...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
      AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

# (b) Margin tiers per (warehouse_id, product_id) - margin VALUES, not prices.
print('Loading margin tiers...')
df_margin_tiers = get_margin_tiers()
print(f'  Margin tiers: {len(df_margin_tiers)} rows')

print('\nAll data loaded.')

In [ ]:
# =============================================================================
# CELL 3 - BUILD LOOKUP (product x cohort) + margin-tier merge
# =============================================================================

# Step 3a - same shape as manual_price_push lookup
df_market_cohorts = expand_to_cohorts(market_data)

lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# brand x cat target margin fallback (and 0.05 default)
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_margin_tgt', 'target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns
                     if c.startswith('target_bm') or c.startswith('cat_target') or c == 'target_margin_tgt'],
            inplace=True, errors='ignore')

# market percentile derivatives from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# current price per (product, cohort) - basic unit
base_prices = (
    df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']]
    .drop_duplicates(subset=['cohort_id', 'product_id'])
)
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# commercial_min_price per (product, cohort)
if 'cohort_id' in df_commercial_min.columns:
    cmin = df_commercial_min[['product_id', 'cohort_id', 'commercial_min_price']].drop_duplicates()
    lookup = lookup.merge(cmin, on=['product_id', 'cohort_id'], how='left')
else:
    lookup['commercial_min_price'] = np.nan

# total stocks per product (display only)
product_stocks = (
    df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    .rename(columns={'stocks': 'total_stocks'})
)
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With current_price: {lookup["current_price"].notna().sum()}')
print(f'  With commercial_min: {lookup["commercial_min_price"].notna().sum()}')

# Step 3b - margin tiers, warehouse -> cohort by MEAN of margin values
margin_tier_cols = [
    'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2',
]

wh_to_cohort = pd.DataFrame(
    WAREHOUSE_MAPPING,
    columns=['region', 'wh_name', 'warehouse_id', 'cohort_id'],
)[['warehouse_id', 'cohort_id']]

_present_tier_cols = [c for c in margin_tier_cols if c in df_margin_tiers.columns]
df_margin_tiers_cohort = (
    df_margin_tiers.merge(wh_to_cohort, on='warehouse_id', how='inner')
                   .groupby(['cohort_id', 'product_id'], as_index=False)[_present_tier_cols]
                   .mean()
)
lookup = lookup.merge(
    df_margin_tiers_cohort,
    on=['cohort_id', 'product_id'],
    how='left',
)
if 'margin_tier_1' in lookup.columns:
    print(f'  With margin_tier_1: {lookup["margin_tier_1"].notna().sum()}')

In [ ]:
# =============================================================================
# CELL 4 - ACHIEVEMENT JOIN (yesterday)
# Roll up warehouse to cohort by summing yest_qty + p80_target across the
# cohort's warehouses, then recompute qty_ratio at cohort level.
# =============================================================================
ACHIEVMENT_QUERY = open(os.path.join('queries', 'achievment_yasterday.sql'), encoding='utf-8').read()
df_ach = query_snowflake(ACHIEVMENT_QUERY)
print(f'Achievement rows (warehouse, product): {len(df_ach)}')

# achievment_yasterday returns warehouse_id, product_id, ... .
# Roll up to cohort using WAREHOUSE_MAPPING (already loaded above as wh_to_cohort).
#df_ach_w = df_ach.merge(wh_to_cohort, on='warehouse_id', how='inner')

df_ach_cohort = (
    df_ach.groupby(['cohort_id', 'product_id'], as_index=False)
            .agg(yest_qty=('yest_qty', 'sum'),
                 p80_target=('p80_target', 'sum'),
                 opening_stock=('opening_stock', 'sum'),
                 closing_stock=('closing_stock', 'sum'))
)
df_ach_cohort['qty_ratio'] = (
    df_ach_cohort['yest_qty'] /
    df_ach_cohort['p80_target'].replace(0, np.nan)
)

lookup = lookup.merge(df_ach_cohort, on=['cohort_id', 'product_id'], how='left')
lookup['qty_ratio'] = lookup['qty_ratio'].fillna(0)
print(f'Lookup with achievement: {len(lookup)} rows; '
      f'{lookup["closing_stock"].notna().sum()} have closing_stock')

In [ ]:
# =============================================================================
# CELL 4b - 30-DAY SALES-BY-POSITION HISTORY (manual NMV target)
# Pull historical price_position snapshots from MATERIALIZED_VIEWS.Pricing_data_extraction
# and join to daily sales from product_sales_order, then roll up to cohort grain.
# =============================================================================

# >>> EDIT ME: today's national NMV target in EGP <<<
NMV_TARGET = 0.0

# Position-label normalization map (extraction strings -> our labels)
POSITION_LABEL_MAP = {
    'Below Market': 'below_min',
    'Below Min':    'below_min',
    'At Min':       'min',
    'At 25th':      '25',
    'At 50th':      '50',
    'At 75th':      '75',
    'At Max':       'max',
    'Above Market': 'max',
    'No Market Data': 'Target',
}

print(f'NMV target for today: {NMV_TARGET:,.0f} EGP')

print('Loading 30d price_position snapshots from Pricing_data_extraction...')
EXTRACTION_HISTORY_QUERY = f'''
SELECT
    run_date::DATE AS sale_date,
    warehouse_id,
    product_id,
    price_position
FROM MATERIALIZED_VIEWS.Pricing_data_extraction
WHERE run_date::DATE >= DATEADD(day, -30,
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
  AND run_date::DATE <  CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
  AND price_position IS NOT NULL
'''
df_pos_hist = query_snowflake(EXTRACTION_HISTORY_QUERY)
print(f'  Snapshots: {len(df_pos_hist)} rows')

print('Loading 30d daily sales (product_sales_order)...')
DAILY_SALES_QUERY = f'''
WITH parent_whs AS (SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)),
raw AS (
    SELECT pso.product_id,
           pso.warehouse_id,
           CAST(DATE_TRUNC('day', pso.created_at) AS DATE) AS sale_date,
           SUM(pso.purchased_item_count * pso.basic_unit_count) AS qty,
           SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= DATEADD(day, -30,
            CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
      AND so.created_at <  CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
      AND so.sales_order_status_id NOT IN (7, 12)
      AND so.channel IN ('telesales', 'retailer')
      AND pso.purchased_item_count <> 0
    GROUP BY 1, 2, 3
)
SELECT product_id,
       COALESCE(p.parent_id, raw.warehouse_id) AS warehouse_id,
       sale_date,
       SUM(qty) AS qty,
       SUM(nmv) AS nmv
FROM raw
LEFT JOIN parent_whs p ON p.child_id = raw.warehouse_id
GROUP BY 1, 2, 3
'''
df_sales_hist = query_snowflake(DAILY_SALES_QUERY)
print(f'  Daily sales: {len(df_sales_hist)} rows')

# Tag each day's sales with that day's position
df_hist = df_sales_hist.merge(
    df_pos_hist, on=['sale_date', 'warehouse_id', 'product_id'], how='inner'
)
print(f'  Joined sales x position: {len(df_hist)} rows')

# Normalize position label
df_hist['price_position'] = df_hist['price_position'].map(POSITION_LABEL_MAP).fillna(df_hist['price_position'])
unmapped = df_hist[~df_hist['price_position'].isin(['min','25','50','75','max','below_min','Target'])]
if len(unmapped) > 0:
    print(f'  WARNING: {len(unmapped)} rows have unmapped price_position labels:')
    print(unmapped['price_position'].value_counts().head(10))

# Roll up warehouse -> cohort
df_hist = df_hist.merge(wh_to_cohort, on='warehouse_id', how='inner')

# (a) sales-by-position table at sku_cohort grain
sales_by_pos = (
    df_hist.groupby(['cohort_id', 'product_id', 'price_position'], as_index=False)
           .agg(total_nmv=('nmv', 'sum'),
                total_qty=('qty', 'sum'),
                days=('sale_date', 'nunique'))
)
sales_by_pos['avg_daily_nmv'] = sales_by_pos['total_nmv'] / sales_by_pos['days'].replace(0, np.nan)
print(f'  sales_by_pos: {len(sales_by_pos)} rows')

# (b) 30d totals per (cohort_id, product_id)
nmv_30d = (
    df_hist.groupby(['cohort_id', 'product_id'], as_index=False)
           .agg(nmv_30d=('nmv', 'sum'),
                qty_30d=('qty', 'sum'),
                days_with_sales=('sale_date', 'nunique'))
)
print(f'  nmv_30d: {len(nmv_30d)} rows')

# (c) last-7-day daily NMV avg per (cohort_id, product_id)
if len(df_hist) > 0:
    cutoff7 = df_hist['sale_date'].max() - pd.Timedelta(days=6)
    last7 = (
        df_hist[df_hist['sale_date'] >= cutoff7]
            .groupby(['cohort_id', 'product_id'], as_index=False)['nmv'].sum()
    )
    last7['last7_avg_daily_nmv'] = last7['nmv'] / 7.0
    last7 = last7[['cohort_id', 'product_id', 'last7_avg_daily_nmv']]
else:
    last7 = pd.DataFrame(columns=['cohort_id', 'product_id', 'last7_avg_daily_nmv'])
print(f'  last7: {len(last7)} rows')

In [ ]:
# =============================================================================
# CELL 5 - FILTER
# Keep only SKUs that had inventory at both opening and closing of yesterday.
# Achievement query already filters out warehouses with closing or opening = 0,
# so SKUs with NaN closing/opening here were dropped upstream.
# =============================================================================
before = len(lookup)
lookup = lookup[
    (lookup['closing_stock'].fillna(0) > 0) &
    (lookup['opening_stock'].fillna(0) > 0)
].copy()
print(f'Filtered to closing>0 AND opening>0: {before} -> {len(lookup)}')

In [ ]:
# =============================================================================
# CELL 6 - POSITION + ACH BUCKET
# =============================================================================
def derive_position(row):
    cur = row.get('current_price')
    cmin = row.get('commercial_min_price', 0) or 0
    if pd.isna(cur):
        return 'no_price'
    if cmin > 0 and cur < cmin:
        return 'below_min'
    has_market = (
        isinstance(row.get('price_tiers'), list) and len(row['price_tiers']) > 0
        and pd.notna(row.get('market_min'))
    )
    if not has_market:
        return 'Target'
    if cur >= row['market_max']: return 'max'
    if cur >= row['market_75']:  return '75'
    if cur >= row['market_50']:  return '50'
    if cur >= row['market_25']:  return '25'
    return 'min'

def derive_ach_bucket(ach):
    if pd.isna(ach) or ach < 0.9: return 'low'
    if ach < 2.0:                  return 'mid'
    return 'high'

lookup['position']   = lookup.apply(derive_position, axis=1)
lookup['ach_bucket'] = lookup['qty_ratio'].apply(derive_ach_bucket)

print('Position x Ach distribution:')
print(pd.crosstab(lookup['position'], lookup['ach_bucket'], margins=True))

In [ ]:
# =============================================================================
# CELL 6b - ELASTICITY INPUTS + CLASSIFICATION FLAGS
# nmv_share / bottom-quartile / top50-cumulative flags per (cohort, product),
# plus sku_cohort + (cat, brand) sales-by-position dictionaries with fallback.
# =============================================================================

# nmv_share + bottom-quartile flag within each cohort
nmv_30d['cohort_nmv_total'] = nmv_30d.groupby('cohort_id')['nmv_30d'].transform('sum')
nmv_30d['nmv_share'] = nmv_30d['nmv_30d'] / nmv_30d['cohort_nmv_total'].replace(0, np.nan)
nmv_30d['cohort_q1_nmv'] = nmv_30d.groupby('cohort_id')['nmv_30d'].transform(lambda s: s.quantile(0.25))
nmv_30d['is_bottom_quartile'] = nmv_30d['nmv_30d'] <= nmv_30d['cohort_q1_nmv']

# top-50% cumulative NMV flag within each cohort
nmv_30d = nmv_30d.sort_values(['cohort_id', 'nmv_30d'], ascending=[True, False])
nmv_30d['cum_share'] = nmv_30d.groupby('cohort_id')['nmv_share'].cumsum()
nmv_30d['is_top50_cum'] = nmv_30d['cum_share'] <= 0.50

# Merge classification inputs into lookup
lookup = lookup.merge(
    nmv_30d[['cohort_id', 'product_id', 'nmv_30d', 'nmv_share',
             'is_bottom_quartile', 'is_top50_cum']],
    on=['cohort_id', 'product_id'], how='left',
)
lookup = lookup.merge(last7, on=['cohort_id', 'product_id'], how='left')
lookup['nmv_30d'] = lookup['nmv_30d'].fillna(0)
lookup['nmv_share'] = lookup['nmv_share'].fillna(0)
lookup['last7_avg_daily_nmv'] = lookup['last7_avg_daily_nmv'].fillna(0)
lookup['is_bottom_quartile'] = lookup['is_bottom_quartile'].fillna(False)
lookup['is_top50_cum'] = lookup['is_top50_cum'].fillna(False)

# sales-by-position dictionaries: sku_cohort grain + (cat, brand) fallback
sbp_sku = sales_by_pos.set_index(
    ['cohort_id', 'product_id', 'price_position']
)['avg_daily_nmv'].to_dict()

sbp_cb = (
    sales_by_pos.merge(df_products[['product_id', 'brand', 'cat']],
                       on='product_id', how='left')
                .groupby(['cat', 'brand', 'price_position'], as_index=False)['avg_daily_nmv']
                .mean()
)
sbp_cb_dict = sbp_cb.set_index(['cat', 'brand', 'price_position'])['avg_daily_nmv'].to_dict()

def expected_daily_nmv_at(cohort, product, cat, brand, position):
    """Expected daily NMV for this SKU at a hypothetical position bucket.
    Tries (cohort, product) first; falls back to (cat, brand). Returns None
    if neither has data for that position label."""
    v = sbp_sku.get((cohort, product, position))
    if v is not None and not pd.isna(v):
        return float(v)
    v = sbp_cb_dict.get((cat, brand, position))
    if v is not None and not pd.isna(v):
        return float(v)
    return None

# Diagnostic: how many SKUs have any history we can use
has_sku_hist = lookup.apply(
    lambda r: (r['cohort_id'], r['product_id']) in {(c, p) for (c, p, _) in sbp_sku.keys()},
    axis=1,
)
print(f'Lookup rows with sku_cohort history: {has_sku_hist.sum()} / {len(lookup)}')

In [ ]:
# =============================================================================
# CELL 7 - ACTION MATRIX
#   - build_effective_tiers: price_tiers if non-empty, else margin tier prices.
#   - step_in_tiers: move N steps relative to current_price (clamps to ends).
#   - apply_margin_step_bounds: 0.1*tgt <= |delta_margin| <= 0.5*tgt.
#   - compute_action: per-row matrix dispatch.
# Safety nets at the end: 0.9*wac_p hard floor, then commercial_min_price.
# =============================================================================
def build_effective_tiers(row):
    """Sorted ascending list of prices (0.25 EGP rounded).
    Per design: margin MEANs are pre-aggregated at cohort grain in Cell 3, then
    converted to ONE price per tier via wac/(1-mean_margin)."""
    pt = row.get('price_tiers')
    if isinstance(pt, list) and len(pt) > 0:
        return sorted({round(p * 4) / 4 for p in pt if p > 0})
    wac = row.get('wac_p', 0) or 0
    if wac <= 0:
        return []
    margins = []
    for c in ['margin_tier_1','margin_tier_2','margin_tier_3','margin_tier_4',
              'margin_tier_5','margin_tier_above_1','margin_tier_above_2']:
        m = row.get(c)
        if pd.notna(m) and 0 < m < 1:
            margins.append(m)
    return sorted({round(wac / (1 - m) * 4) / 4 for m in margins})


def step_in_tiers(tiers, current, n):
    """Returns (new_price, was_clamped, no_tier_in_direction)."""
    if n == 0 or not tiers:
        return None, False, False
    if n > 0:
        above = [t for t in tiers if t > current]
        if not above:
            return None, False, True
        idx = min(n - 1, len(above) - 1)
        return above[idx], (idx < n - 1), False
    below = [t for t in tiers if t < current][::-1]
    if not below:
        return None, False, True
    idx = min(abs(n) - 1, len(below) - 1)
    return below[idx], (idx < abs(n) - 1), False


def apply_margin_step_bounds(current, candidate, wac, target_margin):
    """Clamp |delta_margin| to [0.1*tgt, 0.5*tgt]. Returns (price, bound_hit)."""
    if candidate is None or wac <= 0 or current <= 0:
        return candidate, None
    cur_m = (current   - wac) / current
    new_m = (candidate - wac) / candidate
    delta = new_m - cur_m
    if delta == 0:
        return candidate, None
    direction = 1 if delta > 0 else -1
    abs_delta = abs(delta)
    min_step  = 0.1 * (target_margin or 0)
    max_step  = 0.5 * (target_margin or 0)
    if min_step <= 0 and max_step <= 0:
        return candidate, None
    if abs_delta < min_step:
        clamped_m = cur_m + direction * min_step
        bound = 'min'
    elif abs_delta > max_step:
        clamped_m = cur_m + direction * max_step
        bound = 'max'
    else:
        return candidate, None
    if clamped_m >= 0.99:
        return candidate, None
    new_price = wac / (1 - clamped_m)
    return round(new_price * 4) / 4, bound


ACTION_STEPS = {
    ('min', 'low'):  0,    # 1
    ('min', 'mid'):  0,    # 2
    ('min', 'high'): +2,   # 3
    ('25',  'low'):  -1,   # 4
    ('25',  'mid'):  -1,   # 5
    ('25',  'high'): +1,   # 6
    ('50',  'low'):  -2,   # 7
    ('50',  'mid'):  -1,   # 8
    ('50',  'high'): +1,   # 9
    ('75',  'low'):  -3,   # 10
    ('75',  'mid'):  -3,   # 11
    ('75',  'high'): 0,    # 12
    ('max', 'low'):  -3,   # 13
    ('max', 'mid'):  -2,   # 14
    ('max', 'high'): 0,    # 15
}


def compute_action(row):
    pos  = row['position']
    ach  = row['ach_bucket']
    cur  = row['current_price']
    cmin = row.get('commercial_min_price', 0) or 0
    wac  = row.get('wac_p', 0) or 0
    tm   = row.get('target_margin', 0.05) or 0.05

    if pos == 'no_price':
        return None, 'no_action_no_price', None
    if pos == 'below_min':
        return round(cmin * 4) / 4, 'lift_to_commercial_min', None

    tiers = build_effective_tiers(row)

    if pos == 'Target':
        cur_margin = (cur - wac) / cur if cur > 0 and wac > 0 else 0
        margin_below = cur_margin < tm
        if margin_below:
            n = +2 if ach == 'high' else -2
            base_label = f'target_marg_below_{ach}_{n:+d}'
        else:
            n = {'low': -2, 'mid': -1, 'high': 0}[ach]
            base_label = (f'target_marg_ok_{ach}_{n:+d}'
                          if n != 0 else f'target_marg_ok_{ach}_hold')
    else:
        n = ACTION_STEPS.get((pos, ach), None)
        base_label = (f'pos_{pos}_{ach}_{n:+d}'
                      if n not in (None, 0) else f'pos_{pos}_{ach}_hold')

    if n in (None, 0):
        return None, base_label, n

    candidate, clamped, no_tier = step_in_tiers(tiers, cur, n)
    if no_tier:
        return None, f'{base_label}_no_tier_{"above" if n > 0 else "below"}', n

    label = base_label + ('_clamped' if clamped else '')

    bounded, bound_hit = apply_margin_step_bounds(cur, candidate, wac, tm)
    if bound_hit:
        label += f'_step_{bound_hit}_bound'
        candidate = bounded

    return candidate, label, n


results = lookup.apply(compute_action, axis=1, result_type='expand')
results.columns = ['new_price', 'reason', 'steps']
lookup = pd.concat(
    [lookup.drop(columns=['new_price', 'reason', 'steps'], errors='ignore'), results],
    axis=1,
)

# Safety net 1: never go below 0.9 * wac_p
mask = lookup['new_price'].notna() & (lookup['wac_p'] > 0)
wac_floor = 0.9 * lookup['wac_p']
below_wac = mask & (lookup['new_price'] < wac_floor)
lookup.loc[below_wac, 'new_price'] = (wac_floor[below_wac] * 4).round() / 4

# Safety net 2: commercial_min takes precedence when present
mask  = lookup['new_price'].notna() & (lookup['commercial_min_price'].fillna(0) > 0)
below = mask & (lookup['new_price'] < lookup['commercial_min_price'])
lookup.loc[below, 'new_price'] = lookup.loc[below, 'commercial_min_price'].round(2)

print('Reason distribution (top 25):')
print(lookup['reason'].value_counts().head(25))

In [ ]:
# =============================================================================
# CELL 7b - CLASSIFICATION (informational columns only)
# This cell only adds the sales_or_margin + reason_v2 informational labels for
# review purposes. The actual price decisions are produced by the optimizer in
# Cell 7c. The matrix output from Cell 7 (new_price, reason, steps) is left
# untouched.
# =============================================================================

def classify_row(row):
    if pd.isna(row.get('nmv_30d')) or row['nmv_30d'] <= 0:
        return 'none', 'none_no_30d_sales'
    if (bool(row.get('is_bottom_quartile'))
            and row['position'] in ('min', '25')
            and float(row.get('qty_ratio', 0) or 0) > 1.5):
        return 'margin', 'margin_eligible'
    if bool(row.get('is_top50_cum')):
        return 'sales', 'sales_eligible'
    return 'none', 'none_not_in_pool'

_class = lookup.apply(
    lambda r: pd.Series(classify_row(r), index=['sales_or_margin', 'reason_v2']),
    axis=1,
)
lookup[['sales_or_margin', 'reason_v2']] = _class
lookup['new_price_v2']    = np.nan
lookup['delta_nmv_v2']    = np.nan
lookup['delta_profit_v2'] = np.nan

print('sales_or_margin distribution (informational):')
print(lookup['sales_or_margin'].value_counts(dropna=False))

In [ ]:
# =============================================================================
# CELL 7c - PROFIT-MAXIMIZING OPTIMIZER
# Goal: hit NMV_TARGET with as little margin loss as possible (and capture extra
#       profit where possible). Three-phase greedy on a candidate-move table.
#
# For each SKU, enumerate steps -3..+3 in effective_tiers (skipping current).
# For each candidate, compute:
#     candidate_price (must satisfy floor max(0.9*wac_p, commercial_min)
#                      and ceiling market_max for SKUs with market data)
#     candidate_position (bucketed by market percentiles)
#     exp_nmv_new    = sales_by_pos[(sku,cohort,new_pos)]   (cat_brand fallback)
#     exp_margin_new = (candidate_price - wac) / candidate_price
#     exp_profit_new = exp_nmv_new * exp_margin_new
#     delta_nmv      = exp_nmv_new    - exp_nmv_cur
#     delta_profit   = exp_profit_new - exp_profit_cur
#
# Phases:
#   1) Free-profit moves: delta_profit > 0 AND delta_nmv >= 0
#      Take the best-by-delta_profit candidate per SKU; pure win.
#   2) Margin-up at NMV cost: delta_profit > 0 AND delta_nmv < 0
#      Only run if we already have NMV slack vs target (gap_after_phase1 < 0).
#      Sort candidates by delta_profit / |delta_nmv|  (profit per NMV sacrificed).
#      Greedy until slack would go negative; one move per SKU.
#   3) NMV-up at profit cost: delta_nmv > 0 AND delta_profit < 0
#      Only run if gap remains. Sort by delta_nmv / |delta_profit| (NMV per
#      profit unit lost). Greedy until gap closed; one move per SKU.
# =============================================================================

def position_of_price(row, candidate):
    """Bucket a hypothetical price into our position labels using the row's
    market percentiles. Returns 'Target' if the SKU has no market data."""
    if not isinstance(row.get('price_tiers'), list) or len(row['price_tiers']) == 0 \
            or pd.isna(row.get('market_min')):
        return 'Target'
    if candidate >= row['market_max']: return 'max'
    if candidate >= row['market_75']:  return '75'
    if candidate >= row['market_50']:  return '50'
    if candidate >= row['market_25']:  return '25'
    return 'min'

# ----- Build candidate-move table over the full lookup -----
candidates = []
for idx, row in lookup.iterrows():
    cur = row.get('current_price')
    wac = float(row.get('wac_p', 0) or 0)
    if pd.isna(cur) or wac <= 0:
        continue
    cmin = float(row.get('commercial_min_price', 0) or 0)
    floor = max(0.9 * wac, cmin)
    has_market = isinstance(row.get('price_tiers'), list) and len(row['price_tiers']) > 0 \
                 and pd.notna(row.get('market_max'))
    ceiling = float(row['market_max']) if has_market else float('inf')

    tiers = build_effective_tiers(row)
    if not tiers:
        continue

    cur_pos = row['position']
    exp_nmv_cur = expected_daily_nmv_at(
        row['cohort_id'], row['product_id'], row.get('cat'), row.get('brand'), cur_pos,
    )
    if exp_nmv_cur is None:
        exp_nmv_cur = float(row.get('last7_avg_daily_nmv', 0) or 0)
    cur_margin = (cur - wac) / cur if cur > 0 else 0
    exp_profit_cur = exp_nmv_cur * cur_margin

    for n in (-3, -2, -1, 1, 2, 3):
        cand, _, _ = step_in_tiers(tiers, cur, n)
        if cand is None:
            continue
        if cand < floor or cand > ceiling:
            continue
        new_pos = position_of_price(row, cand)
        exp_nmv_new = expected_daily_nmv_at(
            row['cohort_id'], row['product_id'], row.get('cat'), row.get('brand'), new_pos,
        )
        if exp_nmv_new is None:
            continue
        new_margin     = (cand - wac) / cand if cand > 0 else 0
        exp_profit_new = exp_nmv_new * new_margin
        candidates.append({
            'idx':           idx,
            'cohort_id':     row['cohort_id'],
            'product_id':    row['product_id'],
            'step':          n,
            'cand_price':    cand,
            'cand_position': new_pos,
            'exp_nmv_cur':   exp_nmv_cur,
            'exp_nmv_new':   exp_nmv_new,
            'delta_nmv':     exp_nmv_new - exp_nmv_cur,
            'delta_profit':  exp_profit_new - exp_profit_cur,
        })

cand_df = pd.DataFrame(candidates)
print(f'Candidate moves built: {len(cand_df)} rows over '
      f'{cand_df["idx"].nunique() if len(cand_df) else 0} SKUs')

# ----- Compute the gap -----
baseline_total = float(lookup['last7_avg_daily_nmv'].sum())
gap = NMV_TARGET - baseline_total   # may be negative (already above target)
print(f'Baseline (sum of last7_avg_daily_nmv): {baseline_total:,.0f}')
print(f'Target:                                 {NMV_TARGET:,.0f}')
print(f'Initial gap:                            {gap:,.0f}')

decided = set()           # SKU idxs already assigned a v2 move
running_dnmv    = 0.0     # accumulated delta NMV from picked moves
running_dprofit = 0.0     # accumulated delta profit from picked moves

def _apply(idx, c):
    lookup.at[idx, 'new_price_v2']    = c['cand_price']
    lookup.at[idx, 'delta_nmv_v2']    = c['delta_nmv']
    lookup.at[idx, 'delta_profit_v2'] = c['delta_profit']
    decided.add(idx)

# ----- Phase 1: free-profit moves (profit up, NMV up or flat) -----
if len(cand_df):
    p1 = cand_df[(cand_df['delta_profit'] > 0) & (cand_df['delta_nmv'] >= 0)]
    p1_best = (p1.sort_values(['delta_profit', 'delta_nmv'], ascending=[False, False])
                 .drop_duplicates(subset=['idx'], keep='first'))
    p1_applied = 0
    for _, c in p1_best.iterrows():
        _apply(c['idx'], c)
        running_dnmv    += c['delta_nmv']
        running_dprofit += c['delta_profit']
        lookup.at[c['idx'], 'reason_v2'] = (
            f'p1_free_profit_step_{int(c["step"]):+d}_to_{c["cand_position"]}'
        )
        p1_applied += 1
    print(f'Phase 1 (free profit) moves applied: {p1_applied}')
else:
    print('No candidates to optimize')

gap_after_p1 = gap - running_dnmv
slack_after_p1 = -gap_after_p1  # positive = above target

# ----- Phase 2: margin-up moves at NMV cost (only if we have slack) -----
if len(cand_df) and slack_after_p1 > 0:
    p2 = cand_df[
        ~cand_df['idx'].isin(decided)
        & (cand_df['delta_profit'] > 0)
        & (cand_df['delta_nmv'] < 0)
    ].copy()
    p2['eff'] = p2['delta_profit'] / (-p2['delta_nmv']).replace(0, np.nan)
    p2 = p2.sort_values('eff', ascending=False)
    p2_applied = 0
    slack = slack_after_p1
    for _, c in p2.iterrows():
        if c['idx'] in decided:
            continue
        if (-c['delta_nmv']) > slack:
            continue   # this move would push us below target; skip
        _apply(c['idx'], c)
        running_dnmv    += c['delta_nmv']
        running_dprofit += c['delta_profit']
        slack += c['delta_nmv']    # delta_nmv is negative so slack shrinks
        lookup.at[c['idx'], 'reason_v2'] = (
            f'p2_margin_up_step_{int(c["step"]):+d}_to_{c["cand_position"]}'
        )
        p2_applied += 1
    print(f'Phase 2 (margin-up using slack) moves applied: {p2_applied}')

gap_after_p2 = gap - running_dnmv

# ----- Phase 3: NMV-up at profit cost (only if gap remains) -----
if len(cand_df) and gap_after_p2 > 0:
    p3 = cand_df[
        ~cand_df['idx'].isin(decided)
        & (cand_df['delta_nmv'] > 0)
        & (cand_df['delta_profit'] < 0)
    ].copy()
    p3['eff'] = p3['delta_nmv'] / (-p3['delta_profit']).replace(0, np.nan)
    p3 = p3.sort_values('eff', ascending=False)
    p3_applied = 0
    remaining = gap_after_p2
    for _, c in p3.iterrows():
        if remaining <= 0:
            break
        if c['idx'] in decided:
            continue
        _apply(c['idx'], c)
        running_dnmv    += c['delta_nmv']
        running_dprofit += c['delta_profit']
        remaining       -= c['delta_nmv']
        lookup.at[c['idx'], 'reason_v2'] = (
            f'p3_close_gap_step_{int(c["step"]):+d}_to_{c["cand_position"]}'
        )
        p3_applied += 1
    print(f'Phase 3 (gap-closing drops) moves applied: {p3_applied}')

# Tag SKUs that had usable history but no chosen move
mask_no_move = (~lookup.index.isin(decided)) & (lookup['nmv_30d'] > 0)
lookup.loc[mask_no_move & (lookup['reason_v2'].isin(['margin_eligible',
                                                     'sales_eligible',
                                                     'none_not_in_pool'])),
           'reason_v2'] = 'v2_held_no_better_move'

# Floors safety net (defense-in-depth; candidates were already filtered)
mask = lookup['new_price_v2'].notna() & (lookup['wac_p'] > 0)
floor_series = np.maximum(0.9 * lookup['wac_p'].fillna(0),
                          lookup['commercial_min_price'].fillna(0))
below = mask & (lookup['new_price_v2'] < floor_series)
if below.any():
    lookup.loc[below, 'new_price_v2'] = floor_series[below].round(2)
    lookup.loc[below, 'reason_v2'] = lookup.loc[below, 'reason_v2'].astype(str) + '_floored'

print()
print(f'Final delta_nmv:    {running_dnmv:,.0f}   '
      f'projected NMV total: {baseline_total + running_dnmv:,.0f}')
print(f'Final delta_profit: {running_dprofit:,.0f}')
print(f'NMV target:         {NMV_TARGET:,.0f}   '
      f'remaining gap: {max(NMV_TARGET - (baseline_total + running_dnmv), 0):,.0f}')
print()
print('reason_v2 distribution:')
print(lookup['reason_v2'].value_counts().head(20))

In [ ]:
# =============================================================================
# CELL 8 - REVIEW EXPORT
# =============================================================================
_matrix_action = (
    lookup['new_price'].notna()
    & (lookup['new_price'] != lookup['current_price'])
)
_v2_action = (
    lookup['new_price_v2'].notna()
    & (lookup['new_price_v2'] != lookup['current_price'])
) if 'new_price_v2' in lookup.columns else pd.Series(False, index=lookup.index)

to_review = lookup[_matrix_action | _v2_action].copy()
to_review['delta']     = to_review['new_price'] - to_review['current_price']
to_review['delta_pct'] = to_review['delta'] / to_review['current_price']
if 'new_price_v2' in to_review.columns:
    to_review['delta_v2']     = to_review['new_price_v2'] - to_review['current_price']
    to_review['delta_pct_v2'] = to_review['delta_v2'] / to_review['current_price']

cols = [
    'cohort_id', 'product_id', 'sku', 'brand', 'cat',
    'wac_p', 'target_margin', 'commercial_min_price',
    'price_tiers',
    'market_min', 'market_25', 'market_50', 'market_75', 'market_max', 'market_avg',
    'current_price', 'qty_ratio', 'position', 'ach_bucket', 'steps',
    'opening_stock', 'closing_stock', 'total_stocks',
    'reason', 'new_price', 'delta', 'delta_pct',
    # v2 sales/margin classification
    'sales_or_margin', 'nmv_30d', 'nmv_share', 'last7_avg_daily_nmv',
    'is_top50_cum', 'is_bottom_quartile',
    'new_price_v2', 'delta_v2', 'delta_pct_v2',
    'delta_nmv_v2', 'delta_profit_v2', 'reason_v2',
]
to_review = to_review[[c for c in cols if c in to_review.columns]]

print(f'Total SKUs with action: {len(to_review)}')
print('Reason distribution:')
print(to_review['reason'].value_counts())
print('\nPosition x Ach distribution:')
print(pd.crosstab(to_review['position'], to_review['ach_bucket'], margins=True))

stamp = datetime.now(CAIRO_TZ).strftime('%Y%m%d_%H%M')
out_path = f'market_position_pricing_review_{stamp}.xlsx'
to_review.to_excel(out_path, index=False, engine='xlsxwriter')
print(f'\nReview file written: {out_path}')